In [1]:
from transformers import AutoImageProcessor, AutoModel, Dinov2WithRegistersForImageClassification
from PIL import Image
import requests

import torch
import torch.nn.functional as F  # noqa: N812
import torchvision
from torch import Tensor, nn
from torchvision.models._utils import IntermediateLayerGetter
from torchvision.ops.misc import FrozenBatchNorm2d

/opt/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set the environment variable to use a specific GPU
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

## Dinov2 registered

### testing model

In [ ]:
# Freeze the whole backbone 
from transformers import AutoBackbone
bb = AutoBackbone.from_pretrained("facebook/dinov2-with-registers-base")

print("num_hidden_layers :", bb.config.num_hidden_layers)   # → 12
print("stage_names       :", bb.stage_names)                # ['stem', 'stage1', … , 'stage12']
print("default indices   :", bb.out_indices)                # usually (12,) if you didn’t override

# Inspect
n_trainable = sum(p.requires_grad for p in bb.parameters())
print(f"{n_trainable} of {sum(1 for _ in bb.parameters())} parameter tensors require grad")

# Freeze everything
bb.requires_grad_(False)              # PyTorch ≥2.0 convenience
#            └─ or explicitly:
# for p in bb.parameters():
#     p.requires_grad = False

# Verify
assert all(not p.requires_grad for p in bb.parameters())

# Inspect
n_trainable = sum(p.requires_grad for p in bb.parameters())
print(f"{n_trainable} of {sum(1 for _ in bb.parameters())} parameter tensors require grad")

num_hidden_layers : 12
stage_names       : ['stem', 'stage1', 'stage2', 'stage3', 'stage4', 'stage5', 'stage6', 'stage7', 'stage8', 'stage9', 'stage10', 'stage11', 'stage12']
default indices   : [12]
224 of 224 parameter tensors require grad
0 of 224 parameter tensors require grad


### Let model run through image

In [19]:
from transformers import AutoBackbone, AutoImageProcessor

dinov2_ckpt = "facebook/dinov2-with-registers-base"          # ViT-B/16, 768-dim
image_proc  = AutoImageProcessor.from_pretrained(dinov2_ckpt)

# Load an image from the web
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)


# Load the backbone model
backbone = AutoBackbone.from_pretrained(       # returns feature maps already
    dinov2_ckpt,
    out_indices=(12,),        # last transformer block
    reshape_hidden_states=True,  # gives B,C,H,W instead of sequence 
)

pixel_values = image_proc(images=image, return_tensors="pt")["pixel_values"]
img_features = backbone(pixel_values).feature_maps[0]   # B,768,16,16
print(img_features.shape)  # torch.Size([1, 768, 16, 16])

print(backbone.config.hidden_size)



torch.Size([1, 768, 16, 16])
768


## Swin Transformer V2

In [21]:
# Freeze the whole backbone 
from transformers import AutoBackbone
bb = AutoBackbone.from_pretrained("microsoft/swinv2-tiny-patch4-window8-256")

print("num_hidden_layers :", bb.config.num_hidden_layers)   # → 4
print("stage_names       :", bb.stage_names)                # ['stem', 'stage1', … , 'stage4']
print("default indices   :", bb.out_indices)                # usually (4,) if you didn’t override

# Inspect
n_trainable = sum(p.requires_grad for p in bb.parameters())
print(f"{n_trainable} of {sum(1 for _ in bb.parameters())} parameter tensors require grad")

# Freeze everything
bb.requires_grad_(False)              # PyTorch ≥2.0 convenience
#            └─ or explicitly:
# for p in bb.parameters():
#     p.requires_grad = False

# Verify
assert all(not p.requires_grad for p in bb.parameters())

# Inspect
n_trainable = sum(p.requires_grad for p in bb.parameters())
print(f"{n_trainable} of {sum(1 for _ in bb.parameters())} parameter tensors require grad")

num_hidden_layers : 4
stage_names       : ['stem', 'stage1', 'stage2', 'stage3', 'stage4']
default indices   : [4]
241 of 241 parameter tensors require grad
0 of 241 parameter tensors require grad


In [7]:
from transformers import AutoImageProcessor, Swinv2ForMaskedImageModeling
import torch
from PIL import Image
import requests

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

SwinTv2_ckpt = "microsoft/swinv2-tiny-patch4-window8-256" 

image_processor = AutoImageProcessor.from_pretrained(SwinTv2_ckpt)

# pick the stage you want – stage 4 is the deepest / highest-level
backbone = AutoBackbone.from_pretrained(       # returns feature maps already
    SwinTv2_ckpt,
    out_indices=(4,),  # last transformer block
)

pixel_values = image_proc(images=image, return_tensors="pt")["pixel_values"]
img_features = backbone(pixel_values).feature_maps[0]   # B,768,7,7
print(img_features.shape)  # torch.Size([1, 768, 7, 7])

print(backbone.config.hidden_size)

torch.Size([1, 768, 7, 7])
768


In [ ]:

# Load an image from the web
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
model = AutoModel.from_pretrained('facebook/dinov2-base')

# Accessing the model configuration
# configuration = model.config
# print(configuration)

inputs = processor(images=image, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs)
    
# print(logits)  # (batch_size, num_classes)

In [5]:
vision_backbone =  "resnet18"  # Default backbone model
replace_final_stride_with_dilation = False  # Default setting for stride replacement
pretrained_backbone_weights: str | None = "ResNet18_Weights.IMAGENET1K_V1"

backbone_model = getattr(torchvision.models, vision_backbone)(
                replace_stride_with_dilation=[False, False, replace_final_stride_with_dilation],
                weights=pretrained_backbone_weights,
                norm_layer=FrozenBatchNorm2d,
            )
            # Note: The assumption here is that we are using a ResNet model (and hence layer4 is the final
            # feature map).
            # Note: The forward method of this returns a dict: {"feature_map": output}.
            
backbone = IntermediateLayerGetter(backbone_model, return_layers={"layer4": "feature_map"})
fake_img = torch.randn(64, 3, 480, 640)
img_features = backbone(fake_img)["feature_map"]
print(img_features.shape)  # [1, 512, 14, 14]

torch.Size([64, 512, 15, 20])
